In [4]:
from langchain_text_splitters import CharacterTextSplitter

text_spliter=CharacterTextSplitter(
    separator="\n",
    chunk_size=35,
    chunk_overlap=4
)
text="This is the\n text I would like to chunk up.\n It is the example text for this exercise"
text_spliter.create_documents([text])


[Document(metadata={}, page_content='This is the'),
 Document(metadata={}, page_content='text I would like to chunk up.'),
 Document(metadata={}, page_content='It is the example text for this exercise')]

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splite=RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""],
    chunk_size=10,
    chunk_overlap=4,
)

text= "This is\n\n the\n\n text\n I would like\n to chunk up. It is the example text\n for\n\n this exercise"
text_splite.create_documents([text])


[Document(metadata={}, page_content='This is'),
 Document(metadata={}, page_content='the'),
 Document(metadata={}, page_content='text'),
 Document(metadata={}, page_content='I would'),
 Document(metadata={}, page_content='like'),
 Document(metadata={}, page_content='to chunk'),
 Document(metadata={}, page_content='up. It is'),
 Document(metadata={}, page_content='is the'),
 Document(metadata={}, page_content='example'),
 Document(metadata={}, page_content='text'),
 Document(metadata={}, page_content='for'),
 Document(metadata={}, page_content='this'),
 Document(metadata={}, page_content='exercise')]

In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_document="# Intro \n\n    ## History \n\n context \n\n Markdown[9] is a lightweight markup language for creating formatted text using a plain-text editor. John Gruber created Markdown in 2004 as a markup language that is appealing to human readers in its source code form.[9] \n\n Markdown is widely used in blogging, instant messaging, online forums, collaborative software, documentation pages, and readme files. \n\n ## Rise and divergence \n\n As Markdown popularity grew rapidly, many Markdown implementations appeared, driven mostly by the need for \n\n additional features such as tables, footnotes, definition lists,[note 1] and Markdown inside HTML blocks. \n\n #### Standardization \n\n From 2012, a group of people, including Jeff Atwood and John MacFarlane, launched what Atwood characterised as a standardisation effort. \n\n ## Implementations \n\n Implementations of Markdown are available for over a dozen programming languages."
headers_split=[
    ("#","Header 1"),
    ("##","Header 2"),
    ("###","Header 3"),
]
markdown_split=MarkdownHeaderTextSplitter(headers_to_split_on=headers_split)
md_header_split=markdown_split.split_text(markdown_document)
md_header_split

[Document(metadata={'Header 1': 'Intro', 'Header 2': 'History'}, page_content='context  \nMarkdown[9] is a lightweight markup language for creating formatted text using a plain-text editor. John Gruber created Markdown in 2004 as a markup language that is appealing to human readers in its source code form.[9]  \nMarkdown is widely used in blogging, instant messaging, online forums, collaborative software, documentation pages, and readme files.'),
 Document(metadata={'Header 1': 'Intro', 'Header 2': 'Rise and divergence'}, page_content='As Markdown popularity grew rapidly, many Markdown implementations appeared, driven mostly by the need for  \nadditional features such as tables, footnotes, definition lists,[note 1] and Markdown inside HTML blocks.  \n#### Standardization  \nFrom 2012, a group of people, including Jeff Atwood and John MacFarlane, launched what Atwood characterised as a standardisation effort.'),
 Document(metadata={'Header 1': 'Intro', 'Header 2': 'Implementations'}, pa

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
recursive_split=RecursiveCharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=4,
    separators=["\n\n","\n"," ",""],
)  #只分割page_content部分，当分割出多份时，会复制metadata部分去生成新的Document
splits = recursive_split.split_documents(md_header_split)
#split_documents传入的是文件，create_document传入的是字符串
for i, d in enumerate(splits[:5]):
    print(f"\n--- chunk {i} ---")
    print(d.metadata)
    print(d.page_content)



--- chunk 0 ---
{'Header 1': 'Intro', 'Header 2': 'History'}
context

--- chunk 1 ---
{'Header 1': 'Intro', 'Header 2': 'History'}
Markdown[9] is a

--- chunk 2 ---
{'Header 1': 'Intro', 'Header 2': 'History'}
a lightweight

--- chunk 3 ---
{'Header 1': 'Intro', 'Header 2': 'History'}
markup language for

--- chunk 4 ---
{'Header 1': 'Intro', 'Header 2': 'History'}
for creating


In [8]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

code_path="/hard_data1/user/yangguobin/LLM/Langchain/ RAG/code.py"
docs=TextLoader(code_path,encoding="utf-8").load()

python_split=RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=300,
    chunk_overlap=30,
)

code_split=python_split.split_documents(docs)

for i ,d in enumerate(code_split):
    d.metadata["chunk_id"]=i

vectorstore=FAISS.from_documents(code_split,embeddings)
retriver=vectorstore.as_retriever(search_kwargs={"k":4})

#把检索到的多个document给整合起来
def format_docs(docs):
    return "\n\n".join(
        [f"[source={d.metadata.get('source')} chunk={d.metadata.get('chunk_id')}]\n {d.page_content}" for d in docs]
    )


prompt=ChatPromptTemplate.from_messages(
    [
        ("system","你是代码助手。只根据上下文回答；如果上下文不足，请说不知道。回答时尽量指出参考的 source/chunk。"),
        ("human","问题:{question}\n\n上下文:{context}")
    ]
)


from langchain_deepseek import ChatDeepSeek
chat=ChatDeepSeek(
    model="deepseek-chat",   # DeepSeek-V3: 支持 tools/structured output
    temperature=0,
    api_key="sk-3d4dbf63d7704840976b05ddbb9957a7"
)



rag_chain=(
    {
       "context":retriver|format_docs,
       "question":RunnablePassthrough(), #所有分支默认接受相同的输入，确保了question和context初始接受到的都是invoke输入的内容
    }
    |prompt
    |chat
)

answer=rag_chain.invoke("这个文件里 hello_world 函数做了什么？")
print(answer)

content='根据提供的上下文，`hello_world` 函数的功能是打印一条问候消息。具体来说，它执行 `print("Hello, World!")` 来输出 "Hello, World!" 到控制台。\n\n参考 source/chunk=0 中的代码片段：\n```python\ndef hello_world():\n    """Print a hello message."""\n    print("Hello, World!")\n```\n\n此外，在 source/chunk=6 中，当脚本作为主程序运行时，会调用 `hello_world()` 函数。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 337, 'total_tokens': 444, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 17}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'c61cf5cc-192d-460d-b306-e899da688a22', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cb266-a9fd-74e2-a1d5-b54cf588d2aa-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 337, 'output_tokens': 107, 'total_tokens': 444, 'input_token_de